In [1]:
import json, pathlib, itertools, math
import pandas as pd

BASE = pathlib.Path.cwd()
DATA = BASE / "data"
ART = BASE / "artifacts"
ART.mkdir(exist_ok=True)

def load_json_safe(path):
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"[warn] Could not load {path}: {e}")
        return {}

# artifacts from Notebook 01
legacy_map = load_json_safe(DATA / "ncbi_protein_legacy_map.json")
seed_classification = load_json_safe(ART / "seed_classification.json")
eti_list = set(load_json_safe(ART / "eti_list.json"))
ivacaftor_list = set(load_json_safe(ART / "ivacaftor_list.json"))

# genotype meta
genotype = load_json_safe(DATA / "Genotype_variant.json")
geno = genotype.get("genotypes", {})
var_prev = geno.get("variant_prevalence", {})
mod_lists = geno.get("modulator_eligibility_lists", {})

print("legacy_map entries:", len(legacy_map))
print("seed_classification:", len(seed_classification))
print("ETI list:", len(eti_list), "Ivacaftor list:", len(ivacaftor_list))
print("variant_prevalence keys:", len(var_prev))

def standardize_variant(v):
    if v is None:
        return None
    return legacy_map.get(v, v)

def get_pathogenicity(v):
    v = standardize_variant(v)
    if not v:
        return "uncertain"
    if v in seed_classification:
        return seed_classification[v]
    if v in ivacaftor_list:
        return "responsive_ivacaftor"
    if v in eti_list:
        return "responsive_ETI"
    return "uncertain"

def is_pathogenic_class(c):
    return c in ["pathogenic", "responsive_ivacaftor", "responsive_ETI"]

def pathogenic_count(v1, v2):
    return sum(is_pathogenic_class(get_pathogenicity(v)) for v in [v1, v2])

def get_modulator_group(v1, v2):
    v1s, v2s = standardize_variant(v1), standardize_variant(v2)
    vs = {v1s, v2s}
    if "F508del" in vs and len(vs) == 1:
        return 1
    if "F508del" in vs:
        return 2
    if (v1s in eti_list) or (v2s in eti_list):
        return 3
    if (v1s in ivacaftor_list) or (v2s in ivacaftor_list):
        return 4
    return 5

def classify_cf(v1, v2):
    c1, c2 = get_pathogenicity(v1), get_pathogenicity(v2)
    if is_pathogenic_class(c1) and is_pathogenic_class(c2):
        return "CF positive"
    if (is_pathogenic_class(c1) and c2 == "uncertain") or (is_pathogenic_class(c2) and c1 == "uncertain"):
        return "risk uncertain"
    return "CF negative / carrier"

# Build variant universe
universe = set(var_prev.keys()) | set(eti_list) | set(ivacaftor_list) | set(seed_classification.keys())

# Standardize and clean
universe = sorted({standardize_variant(v) for v in universe if v})

print("Total unique variants in universe:", len(universe))

# Control the scope
MAX_VARIANTS = 200  # set to None for full set (may be slow/large)
variants_scored = universe[:MAX_VARIANTS] if MAX_VARIANTS else universe
print("Using variants:", len(variants_scored))

# Generate all unordered pairs with replacement (A/A allowed)
pairs = []
for i, v1 in enumerate(variants_scored):
    for j in range(i, len(variants_scored)):
        v2 = variants_scored[j]
        pairs.append((v1, v2))

print("Total pairs to score:", len(pairs))

rows = []
for v1, v2 in pairs:
    c1 = get_pathogenicity(v1)
    c2 = get_pathogenicity(v2)
    pc = pathogenic_count(v1, v2)
    grp = get_modulator_group(v1, v2)
    risk = classify_cf(v1, v2)
    rows.append({
        "variant1": v1,
        "variant2": v2,
        "class1": c1,
        "class2": c2,
        "pathogenic_count": pc,
        "mod_group": grp,
        "risk": risk
    })

df = pd.DataFrame(rows)
print("Scored pairs:", df.shape)
df.head()

def class_bucket(c):
    if c == "uncertain":
        return "uncertain"
    if is_pathogenic_class(c):
        return "pathogenic_like"
    return c

df["bucket1"] = df["class1"].map(class_bucket)
df["bucket2"] = df["class2"].map(class_bucket)

# Sanity rules
violations = []

# 1) Both pathogenic-like but risk ≠ CF positive
violations.append(df[(df["pathogenic_count"] == 2) & (df["risk"] != "CF positive")])

# 2) Exactly one pathogenic-like but risk ≠ risk uncertain
violations.append(df[(df["pathogenic_count"] == 1) & (df["risk"] != "risk uncertain")])

# 3) Zero pathogenic-like but risk ≠ CF negative/carrier
violations.append(df[(df["pathogenic_count"] == 0) & (df["risk"] != "CF negative / carrier")])

violations = pd.concat(violations).drop_duplicates()
print("Sanity rule violations (should be 0):", len(violations))

# Edge-case buckets for review
vus_plus_path = df[(df["pathogenic_count"] == 1)]
all_uncertain = df[(df["bucket1"] == "uncertain") & (df["bucket2"] == "uncertain")]
eti_or_iva_pairs = df[(df["class1"].str.contains("responsive", na=False)) | (df["class2"].str.contains("responsive", na=False))]

summary = {
    "total_pairs": int(df.shape[0]),
    "CF_positive": int((df["risk"]=="CF positive").sum()),
    "risk_uncertain": int((df["risk"]=="risk uncertain").sum()),
    "CF_negative_carrier": int((df["risk"]=="CF negative / carrier").sum()),
    "vus_plus_path_pairs": int(vus_plus_path.shape[0]),
    "all_uncertain_pairs": int(all_uncertain.shape[0]),
    "eti_or_ivacaftor_pairs": int(eti_or_iva_pairs.shape[0]),
    "sanity_violations": int(violations.shape[0])
}
summary

out_dir = ART / "coverage_audit"
out_dir.mkdir(exist_ok=True)

# Full scored matrix (be mindful of size if you run full universe)
df.to_csv(out_dir / "all_pairs_scored.csv", index=False)

# Targeted slices that judges will care about
vus_plus_path.to_csv(out_dir / "vus_plus_pathogenic.csv", index=False)
all_uncertain.to_csv(out_dir / "all_uncertain_pairs.csv", index=False)
eti_or_iva_pairs.to_csv(out_dir / "responsive_variant_pairs.csv", index=False)

# Brief summary
pd.Series(summary).to_csv(out_dir / "summary_metrics.csv")
print("Saved audit outputs to:", out_dir)
summary

import joblib

model_path = ART / "severity_model.joblib"
feature_order_path = ART / "feature_order.json"

if model_path.exists() and feature_order_path.exists():
    model = joblib.load(model_path)
    feature_order = json.loads(feature_order_path.read_text())

    def severity_for_pair(v1, v2, age=-1, sex="M"):
        feats = {
            "pathogenic_count": pathogenic_count(v1, v2),
            "mod_group": get_modulator_group(v1, v2),
            "age": age if age is not None else -1,
            "sex_male": 1 if str(sex).upper().startswith("M") else 0
        }
        X = pd.DataFrame([[feats[k] for k in feature_order]], columns=feature_order)
        return float(model.predict(X)[0])

    # Example: annotate a sample of rows with severity
    sample = df.sample(20, random_state=7).copy()
    sample["severity_index_est"] = sample.apply(lambda r: severity_for_pair(r["variant1"], r["variant2"], age=25, sex="M"), axis=1)
    sample.head(10)
else:
    print("[info] Severity model not found. Run Notebook 02 first.")



legacy_map entries: 710
seed_classification: 6
ETI list: 13 Ivacaftor list: 12
variant_prevalence keys: 10
Total unique variants in universe: 25
Using variants: 25
Total pairs to score: 325
Scored pairs: (325, 7)
Sanity rule violations (should be 0): 0
Saved audit outputs to: /Users/adithyavikram/Documents/Cystic_Fibrosis_Model/artifacts/coverage_audit
